In [0]:
DROP SCHEMA IF EXISTS gold_olist CASCADE;
CREATE SCHEMA IF NOT EXISTS gold_olist;


In [0]:
USE SCHEMA silver_olist

In [0]:
SHOW TABLES

###1. Fact_orders, Fact_order_item, & Fact_payment

##### 1.1. Fact_orders

In [0]:
--View orders table
SELECT * FROM orders limit 5

In [0]:
--View order_items table
SELECT * FROM order_items limit 5

In [0]:
--View payment table
SELECT * FROM order_payments limit 5

In [0]:
CREATE OR REPLACE TABLE gold_olist.fact_order AS
WITH order_details AS --calculate details of each order such as total_items, goods_values, shipping_fee, and payment_amount
  (SELECT order_id, 
  COUNT(product_id) as total_items,
  ROUND(SUM(price),2) as goods_value, 
  ROUND(SUM(freight_value),2) as shipping_fee,
  ROUND(SUM(freight_value+price),2) as payment_amount
  FROM order_items
  GROUP BY order_id),
payment_details AS(--calculate paid value of each order 
  SELECT order_id,
  ROUND(SUM(payment_value),2) as paid
  FROM order_payments
  GROUP BY order_id
),
avg_order_review --Calculate avg review score on each order
AS(
  SELECT order_id,
  AVG(review_score) as avg_review_score
  FROM order_reviews
  GROUP BY order_id
),
new_order_id AS--Left join orders with order_details cte, payment_details ctes, and customers. 
  (SELECT 
      o.order_id,
      c.customer_unique_id,
      o.order_status,
      o.order_purchase_ts,
      o.order_purchase_date,
      o.order_approved_ts,
      o.order_approved_date,
      o.order_delivered_carrier_ts,
      o.order_delivered_carrier_date,
      o.order_delivered_customer_ts,
      o.order_delivered_customer_date,
      o.order_estimated_delivery_date,
      COALESCE(od.total_items,0) AS total_items,
      COALESCE(od.goods_value,0) AS goods_values,
      COALESCE(od.shipping_fee,0) AS shipping_fee,
      COALESCE(od.payment_amount,0) AS payment_amount,
      COALESCE(pd.paid,0) AS paid,
      a.avg_review_score
  FROM orders o
  LEFT JOIN order_details od
  ON o.order_id = od.order_id
  LEFT JOIN payment_details pd
  ON o.order_id = pd.order_id
  LEFT JOIN customers c
  ON o.customer_id = c.customer_id
  LEFT JOIN avg_order_review a
  ON o.order_id = a.order_id
  )
--Remove invalid rows that having paid amount more than payment_amount on each order, it is possible that customer has paid 1 cent more than payment_amount. 
  SELECT * FROM new_order_id 
  WHERE paid<= payment_amount+0.01

In [0]:
--View fact_order table
SELECT COUNT(*) FROM gold_olist.fact_order

####1.2. Fact_order_item

In [0]:
--View number of row in order_items
SELECT COUNT(*) FROM order_items

In [0]:
--View number of rows of order_items after joining with fact_order(only keep valid orders by excluding invalid rows with invalid order_id)
SELECT COUNT(*)
FROM order_items oi
JOIN gold_olist.fact_order fo 
  ON oi.order_id = fo.order_id;


In [0]:
--Load order_items to gold layer as fact_order_item
CREATE OR REPLACE TABLE gold_olist.fact_order_item AS 
SELECT oi.*
FROM order_items oi
JOIN gold_olist.fact_order fo 
  ON oi.order_id = fo.order_id;

####1.3. Fact_payment

In [0]:
--View order_payments
SELECT * FROM order_payments LIMIT 1

In [0]:
SELECT COUNT(*) FROM order_payments

In [0]:
--Just keep valid order payments, load to gold layer
CREATE OR REPLACE TABLE gold_olist.fact_payment AS
SELECT op.* FROM order_payments op
INNER JOIN gold_olist.fact_order fo
  ON op.order_id = fo.order_id

###2. Dim tables

####2.1. dim_customer

In [0]:
--View customers table
SELECT * FROM customers LIMIT 1

In [0]:
--Create dim_customer table, only keep records of customers who made valid orders. 
CREATE OR REPLACE TABLE gold_olist.dim_customer AS
WITH ranked_customer AS (
  SELECT
    c.customer_unique_id,
    c.customer_zip_code_prefix,
    ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY c.customer_unique_id) AS rn
  FROM silver_olist.customers c
  INNER JOIN gold_olist.fact_order fo
    ON c.customer_unique_id = fo.customer_unique_id
)
SELECT customer_unique_id, customer_zip_code_prefix
FROM ranked_customer
WHERE rn = 1;


In [0]:
--Check customer_unique_id again
SELECT 
  COUNT(*), 
  COUNT(distinct customer_unique_id),
  COUNT(*) = COUNT(distinct customer_unique_id) as is_unique
FROM gold_olist.dim_customer

In [0]:
--Check columns of dim_customer
SELECT * FROM gold_olist.dim_customer LIMIT 1

####2.2. dim_seller

In [0]:
SELECT COUNT(*) FROM sellers

In [0]:
select * from sellers limit 1

In [0]:
-- Load valid sellers table to gold layer
CREATE OR REPLACE TABLE gold_olist.dim_seller AS
WITH valid_seller AS 
(SELECT
 s.seller_id,
 s.seller_zip_code_prefix,
 ROW_NUMBER() OVER (PARTITION BY s.seller_id ORDER BY s.seller_id) as rn
FROM 
  sellers s
INNER JOIN gold_olist.fact_order_item fo
  ON s.seller_id = fo.seller_id)
  
SELECT seller_id,seller_zip_code_prefix
FROM valid_seller
WHERE rn=1

####2.3 dim_product

In [0]:
--View table products
SELECT * FROM products limit 5

In [0]:
--Load to gold layer
CREATE OR REPLACE TABLE gold_olist.dim_product AS
SELECT *
FROM products

#### 2.4. dim_location

In [0]:
--Load geolocation table as dim_location to gold layer
CREATE OR REPLACE TABLE gold_olist.dim_location
SELECT * 
FROM geolocation